In [34]:
from langchain.chains import LLMChain
from langchain.chains.router import MultiPromptChain
from langchain.chains.router.llm_router import LLMRouterChain, RouterOutputParser
from langchain.chains.router.multi_prompt_prompt import MULTI_PROMPT_ROUTER_TEMPLATE
from langchain.output_parsers import ResponseSchema, StructuredOutputParser
from langchain.prompts import PromptTemplate
from langchain.chat_models import ChatOpenAI

In [35]:
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

True

In [36]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

In [37]:
# Step 1: Define structured schema for career plan output
plan_schemas = [
    ResponseSchema(name="title", description="The job title the user wants to achieve."),
    ResponseSchema(name="time_frame", description="Time frame to reach the role (e.g., 2 years)."),
    ResponseSchema(name="skills", description="List of skills required.", type="list"),
    ResponseSchema(name="certifications", description="List of recommended certifications.", type="list"),
    ResponseSchema(name="milestones", description="Step-by-step roadmap milestones.", type="list")
]
plan_parser = StructuredOutputParser.from_response_schemas(plan_schemas)
plan_format_instructions = plan_parser.get_format_instructions()

In [38]:
# Step 2: Function to create advisor prompts with embedded format instructions
def structured_template(domain_description):
    return f"""
You are a career advisor for {domain_description}.
Analyze the user's goal and background. Generate a structured career plan with the following fields:

- title: Desired role
- time_frame: Expected duration
- skills: List of must-have skills
- certifications: List of relevant certifications
- milestones: Step-by-step roadmap

{{format_instructions}}


User Input:
{{input}}
"""


In [39]:
# Step 3: Define three specialized career advisors
prompt_infos = [
    {"name": "tech", "description": "technical roles like software engineer, cloud architect, data scientist"},
    {"name": "marketing", "description": "roles in product marketing, digital marketing, brand strategy"},
    {"name": "finance", "description": "finance roles like analyst, accountant, or investment banking"}
]

destination_chains = {}
for info in prompt_infos:
    prompt = PromptTemplate(
    template=structured_template(info["description"]),
    input_variables=["input"],
    partial_variables={"format_instructions": plan_format_instructions}
    )

    destination_chains[info["name"]] = LLMChain(llm=llm, prompt=prompt)


In [40]:

# Step 4: Create the router using MULTI_PROMPT_ROUTER_TEMPLATE
destinations_str = "\n".join([f"{p['name']}: {p['description']}" for p in prompt_infos])
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(destinations=destinations_str)

router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser()
)

router_chain = LLMRouterChain.from_llm(llm, router_prompt)

In [41]:
# Step 5: Combine into MultiPromptChain
multi_prompt_chain = MultiPromptChain(
    router_chain=router_chain,
    destination_chains=destination_chains,
    default_chain=destination_chains["tech"],
    verbose=True
)

In [43]:
# Step 6: Define final motivational message generator
encouragement_prompt = PromptTemplate(
    input_variables=["input", "title", "time_frame", "skills"],
    template="""
You are a motivational career coach. Based on the following career goal and user plan, write an encouraging summary:

Goal: {input}
Target Role: {title}
Time Frame: {time_frame}
Skills Required: {skills}

Encouragement:"""
)
encouragement_chain = LLMChain(llm=llm, prompt=encouragement_prompt)

In [44]:
#Step 7: Function that runs routing + planning + motivation
def run_upgraded_career_advisor(user_input):
    # Route the input manually
    route_result = router_chain.invoke({"input": user_input})
    advisor_name = route_result["destination"]

    selected_chain = destination_chains.get(advisor_name, destination_chains["tech"])

    # Generate the career plan
    chain_output = selected_chain.invoke({"input": user_input})
    structured_plan = plan_parser.parse(chain_output["text"])

    # Generate motivational message
    encouragement = encouragement_chain.invoke({
        "input": user_input,
        "title": structured_plan["title"],
        "time_frame": structured_plan["time_frame"],
        "skills": ", ".join(structured_plan["skills"])
    })["text"]

    return {
        "advisor": advisor_name,
        "plan": structured_plan,
        "encouragement": encouragement.strip()
    }


In [45]:
# Step 8: Test the system with a sample input
test_input = "I want to transition into product marketing within 2 years. I have a background in sales."
output = run_upgraded_career_advisor(test_input)


In [46]:
output

{'advisor': 'marketing',
 'plan': {'title': 'Product Marketing Manager',
  'time_frame': '2 years',
  'skills': ['Market Research',
   'Product Positioning',
   'Competitive Analysis',
   'Go-to-Market Strategy',
   'Cross-functional Collaboration',
   'Data Analysis',
   'Customer Segmentation',
   'Content Creation'],
  'certifications': ['Certified Product Marketing Manager (CPMM)',
   'HubSpot Content Marketing Certification',
   'Google Analytics Certification',
   'Product Management Certificate (e.g., from General Assembly or Pragmatic Institute)'],
  'milestones': ['1. Conduct a self-assessment to identify transferable skills from sales to product marketing.',
   '2. Enroll in a product marketing course or certification program within the first 3 months.',
   '3. Start networking with product marketing professionals through LinkedIn and industry events within 6 months.',
   '4. Gain experience by collaborating with the marketing team in your current role or seeking a marketing-